In [ ]:
%%capture
!pip install datasets

### Imports

In [ ]:
import torch
import torch.nn as nn

from keras.preprocessing.sequence import pad_sequences
import numpy as np

import torch.optim as optim
import time
import random

from tqdm import tqdm

from transformers import AutoModel
from transformers import AutoTokenizer

#Model

In [ ]:
class Classifier(nn.Module):
  def __init__(self, embed_dim, n_labels):
      super(Classifier, self).__init__()

      self.bert = AutoModel.from_pretrained("prajjwal1/bert-tiny")
      self.fc = nn.Linear(embed_dim, n_labels)

  def forward(self, x, mask):

    out = self.bert(x, mask).pooler_output

    out = self.fc(out.float())
    return torch.sigmoid(out)

In [ ]:
# Print the model size
def print_model_size(model):
  param_size = 0
  param_count = 0
  for param in model.parameters():
    param_size += param.nelement() * param.element_size()
    param_count += param.nelement()
  buffer_size = 0
  for buffer in model.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

  size_all_mb = (param_size + buffer_size) / 1024**2
  print('Model params: {:.3f}M'.format(param_count/1e6))
  print('Model size: {:.3f}MB'.format(size_all_mb))

### Load and Preprocess IMDb Dataset

In [ ]:
VOCAB_SIZE = 30522 #512

In [ ]:
from datasets import load_dataset
import sentencepiece as spm
import os

#load dataset
dataset = load_dataset("imdb")
train_data = dataset['train']
test_data = dataset['test']


text_train = train_data.to_dict()["text"]
label_train = train_data.to_dict()["label"]

text_test = test_data.to_dict()["text"]
label_test = test_data.to_dict()["label"]

assert len(text_train) == len(label_train)
assert len(text_test) == len(label_test)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
#Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f'Dictionary size {tokenizer.vocab_size}')
print(f'Vocabulary: {tokenizer.vocab}')
print(f'Encoding results:  {tokenizer.tokenize("this is a phrase that could be commonly found")} -> {tokenizer.encode("this is a phrase that could be commonly found")}')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Dictionary size 30522
Vocabulary: {'speakers': 7492, '##ath': 8988, '##aldi': 24657, 'nikolay': 28494, 'alaska': 7397, 'seeded': 13916, 'specifications': 15480, 'lyrics': 4581, 'splashed': 22055, 'rainy': 16373, 'shudder': 18261, 'superfamily': 24169, 'enthusiasts': 20305, 'crambidae': 21585, '##nail': 25464, 'commercial': 3293, 'closet': 9346, 'utterly': 12580, 'mitch': 11857, 'aviv': 12724, '##ikh': 28209, 'consensus': 10465, 'eco': 17338, 'dorian': 16092, 'pedro': 7707, 'regent': 11315, '##sg': 28745, 'resistant': 13070, 'stopping': 7458, 'wrath': 14532, '##hunter': 25629, 'roof': 4412, 'scoring': 4577, 'highlighting': 20655, '##dden': 17101, 'irina': 25404, '##neo': 23585, 'bafta': 22284, 'ε': 1159, '1817': 12529, 'doctorate': 8972, '##logies': 21615, 'itv': 11858, 'discusses': 15841, 'pcs': 27019, 'andersson': 28643, 'pensions': 22024, 'shakes': 10854, '##ђ': 29758, '[unused129]': 134, 'forming': 5716, '##jo': 5558, 'evolutionary': 12761, '##rup': 21531, '##ish': 4509, 'clement': 

In [ ]:
train_tokens = list(map(lambda t: tokenizer.encode(t, padding= 'max_length')[:512], text_train))
test_tokens = list(map(lambda t: tokenizer.encode(t, padding= 'max_length')[:512], text_test))

In [ ]:
# train_tokens_ids = pad_sequences(train_tokens, maxlen=512, truncating="post", padding="post", dtype="int")
# test_tokens_ids = pad_sequences(test_tokens, maxlen=512, truncating="post", padding="post", dtype="int")

In [ ]:
train_masks = [[float(i > 0) for i in ii] for ii in train_tokens]
test_masks = [[float(i > 0) for i in ii] for ii in test_tokens]

In [ ]:
train_tokens_tensor = torch.tensor(train_tokens)
train_y_tensor = torch.tensor(np.array(label_train).reshape(-1, 1)).float()

test_tokens_tensor = torch.tensor(test_tokens)
test_y_tensor = torch.tensor(np.array(label_test).reshape(-1, 1)).float()

train_masks_tensor = torch.tensor(train_masks)
test_masks_tensor = torch.tensor(test_masks)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

train_dataset = TensorDataset(train_tokens_tensor, train_masks_tensor, train_y_tensor)
train_sampler = RandomSampler(train_dataset)
train_dataloader = DataLoader(train_dataset, sampler=train_sampler, batch_size=16)

test_dataset = TensorDataset(test_tokens_tensor, test_masks_tensor, test_y_tensor)
test_sampler = SequentialSampler(test_dataset)
test_dataloader = DataLoader(test_dataset, sampler=test_sampler, batch_size=16)

### Initializing The Model

In [ ]:
EMBED_DIM = 128
NUM_HEADS = 8
FORWARD_EXPANSION = 1
MAX_LENGTH = 512

#initialize model
classifier = Classifier(VOCAB_SIZE, MAX_LENGTH, EMBED_DIM, NUM_HEADS, FORWARD_EXPANSION)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classifier.to(device)

# print all model parameters with names
for name, param in classifier.named_parameters():
  print(f"{name}: {param.nelement()}")

#print the model size
print_model_size(classifier)



config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

bert.embeddings.word_embeddings.weight: 3906816
bert.embeddings.position_embeddings.weight: 65536
bert.embeddings.token_type_embeddings.weight: 256
bert.embeddings.LayerNorm.weight: 128
bert.embeddings.LayerNorm.bias: 128
bert.encoder.layer.0.attention.self.query.weight: 16384
bert.encoder.layer.0.attention.self.query.bias: 128
bert.encoder.layer.0.attention.self.key.weight: 16384
bert.encoder.layer.0.attention.self.key.bias: 128
bert.encoder.layer.0.attention.self.value.weight: 16384
bert.encoder.layer.0.attention.self.value.bias: 128
bert.encoder.layer.0.attention.output.dense.weight: 16384
bert.encoder.layer.0.attention.output.dense.bias: 128
bert.encoder.layer.0.attention.output.LayerNorm.weight: 128
bert.encoder.layer.0.attention.output.LayerNorm.bias: 128
bert.encoder.layer.0.intermediate.dense.weight: 65536
bert.encoder.layer.0.intermediate.dense.bias: 512
bert.encoder.layer.0.output.dense.weight: 65536
bert.encoder.layer.0.output.dense.bias: 128
bert.encoder.layer.0.output.Laye

### Training

In [ ]:
optimizer = optim.Adam(classifier.parameters(), lr=1e-5)

In [ ]:
criterion = nn.BCELoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion.to(device);

In [ ]:
from tqdm import tqdm


for epoch in range(10):
    classifier.train()
    train_loss = 0
    tqdm_train_loader = tqdm(train_dataloader, desc=f"Epoch {epoch+1}", leave=False)

    for step_num, batch_data in enumerate(tqdm_train_loader):

        token_ids, masks, labels = tuple(t.to(device) for t in batch_data)

        logits = classifier(token_ids, masks)
        # print(logits)
        # print(labels)

        batch_loss = criterion(logits, labels)
        train_loss += batch_loss.item()

        classifier.zero_grad()
        batch_loss.backward()


        nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)
        optimizer.step()

        log_step = 50
        if step_num % log_step == (log_step - 1):
          tqdm_train_loader.set_postfix(loss = train_loss / log_step)
          train_loss = 0


### Evaluation

In [ ]:
classifier.eval()
bert_predicted = []
all_logits = []

tqdm_test_loader = tqdm(test_dataloader, desc=f"Evaluation: ", leave=False)

with torch.no_grad():
    for step_num, batch_data in enumerate(tqdm_test_loader):

        token_ids, masks, labels = tuple(t.to(device) for t in batch_data)

        logits = classifier(token_ids, masks)
        loss = criterion(logits, labels)
        numpy_logits = logits.cpu().detach().numpy()

        bert_predicted += list(numpy_logits[:, 0] > 0.5)
        all_logits += list(numpy_logits[:, 0])

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(label_test, bert_predicted))

              precision    recall  f1-score   support

           0       0.87      0.84      0.85     12500
           1       0.84      0.87      0.86     12500

    accuracy                           0.85     25000
   macro avg       0.85      0.85      0.85     25000
weighted avg       0.85      0.85      0.85     25000



In [ ]:
from sklearn.metrics import matthews_corrcoef
matthews_corrcoef(label_test, bert_predicted)

0.7077205793968064

In [ ]:
# custom_text = "I really liked this film, it was for sure worth watching. The color correction was sublime, It really gave a lot to the film"
custom_text = "This film sucked, I hated it, it was a waste of time. All actors were terrible and even the lights were terrible. Not going for a rewatch"
enc = [1] + sp.encode(custom_text)[:510] + [2]
enc = enc + [0]* (512-len(enc))
enc_tensor = torch.tensor(enc).unsqueeze(0).to(device)
classifier(enc_tensor)

NameError: name 'sp' is not defined